###  Importante activar un entorno de ejecución con GPU (GPU T4)

In [1]:
!pip install transformers accelerate datasets -q
!pip install huggingface_hub -q

In [2]:
import re
import time
import zipfile
import torch
import os

from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from huggingface_hub import login
from google.colab import drive, files
from dotenv import load_dotenv

MAX_NEW_TOKENS = 350
TEMPERATURE = 0.15
TOP_P = 0.9
REPETITION_PENALTY = 1.1

In [ ]:
# cargamos el token de huggingface, en mi caso esta en la carpeta MyDrive
drive.mount('/content/drive')
load_dotenv('/content/drive/MyDrive/.env' )
token = os.getenv("HF_TOKEN")
login(token=token)

# cargamos el modelo, en nuestro caso Gemma2, esto puede tardar un par de minutos
model_id = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id,device_map="auto",torch_dtype=torch.float16,low_cpu_mem_usage=True)
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

### Paraphrasing with Gemma

In [11]:
def load_txtfile(path):
    res = []
    with open(path, encoding="utf-8") as f:
        text = f.read()
    blocks = text.split("\n\n")  # separar por bloques
    for block in blocks:
        # si el bloque esta vacio pasamos
        if not block.strip():
            continue
        parts = block.split("\n", 1)
        # si no hay pregunta y respuesta pasamos al siguiente/ no tiene el formato
        if len(parts) != 2:
            continue
        # guardo por separdo la pregunta y la respuesta
        question, answer = parts
        # si esta completo la añadimos
        if question and answer:
            res.append({"question": question,"answer": answer})

    return res




def save_txtfile(qa_pairs, file_path):
    with open(file_path,"w", encoding="utf-8") as f:
        for pair in qa_pairs:
            question = " ".join(pair["question"].split() )
            answer = " ".join(pair["answer"].split())
            f.write(f"{question}\n")
            f.write(f"{answer}\n\n")


def extract_parafrased_questions(generated_text, original_question):
    if "Paraphrase:" in generated_text:
      text = generated_text.rsplit("Paraphrase:", 1 )[-1]
    else:
      text = generated_text
    # separamos por linea y eliminamos las lineas vacias
    lines = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            lines.append(line)
    if lines:
        q = lines[0]
    else:
        q = ""
  # si no hay preguntao si no es una pregunta o si es muy larga devuelve la original
    if not q or "?" not in q or len(q)> 200:
        return original_question.strip()

    return q.strip()

In [12]:
def parapfrasedPrompt(question):

    return f"""
You are paraphrasing technical questions asked by students.

Your task is to rewrite the question using different wording,
while keeping EXACTLY the same meaning and level of difficulty.

Rules:
- Keep the question at a STUDENT level (not expert).
- Do NOT make the question more technical or more detailed.
- Do NOT introduce new concepts.
- Do NOT change technical terms.
- Do NOT answer the question.
- Keep a similar length to the original.
- Produce ONE paraphrased version only.
- Output ONLY the paraphrased question.

Examples:

Original:
What is the difference between image intensity and image luminance?
Paraphrase:
Can you explain the difference between image intensity and image luminance?

Original:
What is experimental bias?
Paraphrase:
What does experimental bias mean?

Now paraphrase the following question.

Original:
{question}

Paraphrase:
""".strip()

In [ ]:
# le pasamos el generador y el nombre del archivo generado y guardamos el resultado y lo descargamos
def parapfrase_corpus(generator,out_file="qa_paraphrased_questions.txt"):

    uploaded_file =files.upload()
    input_file = next(iter(uploaded_file))
    qa_list = load_txtfile(input_file)
    paraphrasedC =[]

    for pair in qa_list:

        question = pair["question"]
        prompt = parapfrasedPrompt(question)
        out = generator(prompt, do_sample=False)[0]["generated_text"]
        parafrased_question = extract_parafrased_questions(out, question)
        paraphrasedC.append({"question": parafrased_question, "answer": pair["answer"]})

    save_txtfile(paraphrasedC, out_file)
    files.download(out_file)


parapfrase_corpus(generator, out_file="qa_paraphrased_v2.txt")